In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tvDatafeed import TvDatafeed, Interval



In [8]:
# =========================
# PARÂMETROS
# =========================
ticker = "^BVSP" # código do ativo no yahoo
start_date = "2000-01-01"
end_date = None

df = yf.download(
    ticker,
    start=start_date,
    end=end_date,
    progress=False,
    auto_adjust=True,
    back_adjust=True,
    multi_level_index=False,
    
)
df['ret'] = df['Close'].pct_change()
df.dropna(inplace=True)
df


,Close,High,Low,Open,Volume,ret
Date,,,,,,
2000-01-04,15851.00000,16908.0,15851.0,16908.0,0,-0.063733
2000-01-05,16245.00000,16302.0,15350.0,15871.0,0,0.024856
2000-01-06,16107.00000,16499.0,15977.0,16237.0,0,-0.008495
2000-01-07,16309.00000,16449.0,16125.0,16125.0,0,0.012541
2000-01-10,17022.00000,17057.0,16325.0,16325.0,0,0.043718
...,...,...,...,...,...,...
2025-12-16,158578.00000,162482.0,158558.0,162482.0,9920300,-0.024027
2025-12-17,157327.00000,158611.0,156351.0,158578.0,11344300,-0.007889
2025-12-18,157923.00000,158496.0,157124.0,157327.0,7800100,0.003788


In [9]:
custo = 0.01/100
df["position"] = 0



weekly_ret = df["ret"].resample("W-FRI").sum()
weekly_position = np.where(weekly_ret > 0, 1, 0)

weekly_position = pd.Series(
    weekly_position,
    index=weekly_ret.index
).fillna(0)

df["weekly_ret"] = weekly_ret.reindex(
    df.index,
).fillna(0)

df["position"] = weekly_position.reindex(
    df.index,
    method="ffill"
).fillna(0).shift(1)

df["strategy_ret"] = df["position"] * df["ret"]
df["strategy"] = df["strategy_ret"].cumsum()
df["buy_hold"] = df["ret"].cumsum()
df.dropna(inplace=True)
df

,Close,High,Low,Open,Volume,ret,position,weekly_ret,strategy_ret,strategy,buy_hold
Date,,,,,,,,,,,
2000-01-05,16245.00000,16302.0,15350.0,15871.0,0,0.024856,0.0,0.000000,0.000000,0.000000,-0.038877
2000-01-06,16107.00000,16499.0,15977.0,16237.0,0,-0.008495,0.0,0.000000,-0.000000,0.000000,-0.047371
2000-01-07,16309.00000,16449.0,16125.0,16125.0,0,0.012541,0.0,-0.034830,0.000000,0.000000,-0.034830
2000-01-10,17022.00000,17057.0,16325.0,16325.0,0,0.043718,0.0,0.000000,0.000000,0.000000,0.008888
2000-01-11,16573.00000,17197.0,16573.0,17045.0,0,-0.026378,0.0,0.000000,-0.000000,0.000000,-0.017490
...,...,...,...,...,...,...,...,...,...,...,...
2025-12-16,158578.00000,162482.0,158558.0,162482.0,9920300,-0.024027,1.0,0.000000,-0.024027,1.658489,3.169936
2025-12-17,157327.00000,158611.0,156351.0,158578.0,11344300,-0.007889,1.0,0.000000,-0.007889,1.650600,3.162047
2025-12-18,157923.00000,158496.0,157124.0,157327.0,7800100,0.003788,1.0,0.000000,0.003788,1.654389,3.165835


In [ ]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.7, 0.3], 
    vertical_spacing=0.05
)

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["buy_hold"] * 100,
        name="Buy & Hold",
        line=dict(width=2)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["strategy"] * 100,
        name="Strategy",
        line=dict(width=2)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["position"],
        name="Position",
        line=dict(width=2),
    ),
    row=2, col=1
)

fig.update_layout(
    title=f"{ticker} | week move",
    template="plotly_white",
    height=700,
    legend=dict(x=0.01, y=0.99)
)

fig.update_yaxes(
    title_text="Cumulative Return (%)",
    row=1, col=1
)

fig.update_yaxes(
    title_text="Position",
    range=[-0.05, 1.05],
    row=2, col=1
)

fig.update_xaxes(title_text="Date", row=2, col=1)

fig.show()




In [11]:
total_return_bh = df["buy_hold"].iloc[-1]
total_return_strategy = df["strategy"].iloc[-1]

vol_strategy = df["strategy_ret"].std() * np.sqrt(252)
vol_bh = df["ret"].std() * np.sqrt(252)

sharpe_strategy = (
    (df["strategy_ret"].mean() ) / df["strategy_ret"].std()
) * np.sqrt(252)

sharpe_bh = (
    (df["ret"].mean() ) / df["ret"].std()
) * np.sqrt(252)

print("=== RESULTADOS ===")
print(f"Buy & Hold Return: {total_return_bh:.2%}")
print(f"Strategy Return:   {total_return_strategy:.2%}")
print()
print(f"Buy & Hold Vol: {vol_bh:.2%}")
print(f"Strategy Vol:   {vol_strategy:.2%}")
print()
print(f"Buy & Hold Sharpe: {sharpe_bh:.2f}")
print(f"Strategy Sharpe:   {sharpe_strategy:.2f}")


=== RESULTADOS ===
Buy & Hold Return: 316.72%
Strategy Return:   165.79%

Buy & Hold Vol: 26.97%
Strategy Vol:   18.10%

Buy & Hold Sharpe: 0.47
Strategy Sharpe:   0.36
